# 데이터 다운로드 및 정리 노트북

이 노트북은 외부 데이터셋을 **코드로 내려받아** `data/raw_data` 아래에 정리하는 용도입니다.

대상 데이터셋:

- Zenodo: `https://zenodo.org/records/19496480`
- Mendeley Data: `https://data.mendeley.com/datasets/2mwc6x6kwb/1`

설계 기준:

- Git 저장소에는 데이터 파일을 올리지 않는다.
- 실행 시 원격에서 다운로드하고 로컬 폴더에 저장한다.
- 한글 설명이 깨지지 않도록 마크다운 셀에 UTF-8 문자열을 그대로 둔다.
- 외부 패키지 없이 표준 라이브러리만 사용한다.


In [5]:
from __future__ import annotations

import json
import os
import shutil
import zipfile
from pathlib import Path

import requests


def find_project_root(start: Path | None = None) -> Path:
    """pyproject.toml을 기준으로 프로젝트 루트를 찾는다."""
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / 'data'
DOWNLOAD_ROOT = DATA_ROOT / '_downloads'
RAW_ROOT = DATA_ROOT / 'raw_data'

PREDIST_DIR = RAW_ROOT / 'predist_v2'
XAI4HEAT_DIR = RAW_ROOT / 'xai4heat_scada_dataset'

for path in [DATA_ROOT, DOWNLOAD_ROOT, RAW_ROOT, PREDIST_DIR, XAI4HEAT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'DATA_ROOT     = {DATA_ROOT}')
print(f'RAW_ROOT      = {RAW_ROOT}')
print(f'PREDIST_DIR   = {PREDIST_DIR}')
print(f'XAI4HEAT_DIR  = {XAI4HEAT_DIR}')


PROJECT_ROOT = c:\Project3\HeatGrid_Agent
DATA_ROOT     = c:\Project3\HeatGrid_Agent\data
RAW_ROOT      = c:\Project3\HeatGrid_Agent\data\raw_data
PREDIST_DIR   = c:\Project3\HeatGrid_Agent\data\raw_data\predist_v2
XAI4HEAT_DIR  = c:\Project3\HeatGrid_Agent\data\raw_data\xai4heat_scada_dataset


## 1단계: PreDist 데이터 다운로드 및 압축 해제

Zenodo는 `zip` 파일 1개로 배포되어 있으므로, 아래 코드는 브라우저형 세션으로 zip을 내려받은 뒤 `data/raw_data/predist_v2`에 풀어 넣는다.

주의:

- 현재 환경에서는 Zenodo가 403을 반환할 수 있다.
- 그 경우 브라우저로 zip을 수동 다운로드한 뒤 `data/_downloads/predist_dataset.zip`에 넣거나, `PREDIST_ZIP_LOCAL_SOURCE` 환경변수에 경로를 지정한다.

실행 규칙:

- 이미 zip 파일이 있으면 다시 받지 않는다.
- 이미 압축이 풀려 있으면 다시 풀지 않는다.
- 필요하면 `overwrite=True`로 다시 받거나 다시 풀 수 있다.


In [6]:
PREDIST_ZIP_URL = 'https://zenodo.org/records/19496480/files/predist_dataset.zip'
PREDIST_ZIP_PATH = DOWNLOAD_ROOT / 'predist_dataset.zip'
PREDIST_ZIP_LOCAL_SOURCE = os.environ.get('PREDIST_ZIP_LOCAL_SOURCE', '').strip()


def download_file(url: str, destination: Path, *, overwrite: bool = False) -> Path:
    """URL에서 파일을 다운로드한다."""
    if destination.exists() and destination.stat().st_size > 0 and not overwrite:
        print(f'[skip] {destination.name} already exists')
        return destination

    destination.parent.mkdir(parents=True, exist_ok=True)
    session = requests.Session()
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Cache-Control': 'max-age=0',
        'Upgrade-Insecure-Requests': '1',
        'Referer': 'https://zenodo.org/records/19496480',
    }
    session.get('https://zenodo.org/records/19496480', headers=headers, timeout=30)
    with session.get(url, headers=headers, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(destination, 'wb') as out:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    out.write(chunk)

    print(f'[download] {url}')
    print(f'          -> {destination}')
    return destination


def extract_zip(zip_path: Path, target_dir: Path, *, overwrite: bool = False) -> None:
    """zip 파일을 압축 해제한다."""
    if target_dir.exists() and any(target_dir.iterdir()) and not overwrite:
        print(f'[skip] {target_dir} already has files')
        return

    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(target_dir)
    print(f'[extract] {zip_path.name} -> {target_dir}')


if PREDIST_ZIP_LOCAL_SOURCE:
    source_path = Path(PREDIST_ZIP_LOCAL_SOURCE)
    if not source_path.exists():
        raise FileNotFoundError(f'PREDIST_ZIP_LOCAL_SOURCE not found: {source_path}')
    shutil.copy2(source_path, PREDIST_ZIP_PATH)
    print(f'[copy] {source_path} -> {PREDIST_ZIP_PATH}')
else:
    download_file(PREDIST_ZIP_URL, PREDIST_ZIP_PATH)
extract_zip(PREDIST_ZIP_PATH, PREDIST_DIR)


[download] https://zenodo.org/records/19496480/files/predist_dataset.zip
          -> c:\Project3\HeatGrid_Agent\data\_downloads\predist_dataset.zip
[extract] predist_dataset.zip -> c:\Project3\HeatGrid_Agent\data\raw_data\predist_v2


## 2단계: XAI4HEAT SCADA 데이터 다운로드

Mendeley Data는 공개 API에서 파일 목록과 파일별 직접 다운로드 URL을 제공한다.

이 데이터셋은 현재 공개 API 기준으로 개별 CSV 파일들로 제공되므로, ZIP을 풀기보다 **각 CSV를 `data/raw_data/xai4heat_scada_dataset`에 바로 저장**하는 방식이 가장 단순하다.

핵심 흐름:

1. 공개 API에서 파일 메타데이터를 가져온다.
2. 각 파일의 `download_url`을 확인한다.
3. 파일명 그대로 로컬 폴더에 저장한다.

참고 URL:

- 데이터셋 페이지: `https://data.mendeley.com/datasets/2mwc6x6kwb/1`
- 공개 API: `https://data.mendeley.com/public-api/datasets/2mwc6x6kwb`


In [7]:
MENDELEY_DATASET_ID = '2mwc6x6kwb'
MENDELEY_PUBLIC_API = f'https://data.mendeley.com/public-api/datasets/{MENDELEY_DATASET_ID}'


def fetch_json(url: str) -> dict:
    request = urllib.request.Request(
        url,
        headers={
            'User-Agent': 'Mozilla/5.0',
            'Accept': 'application/json',
        },
    )
    with urllib.request.urlopen(request) as response:
        return json.loads(response.read().decode('utf-8'))


def download_mendeley_dataset(dataset_dir: Path, *, overwrite: bool = False) -> None:
    metadata = fetch_json(MENDELEY_PUBLIC_API)
    files = metadata.get('files', [])

    print(f"[mendeley] dataset: {metadata.get('name')}")
    print(f"[mendeley] files  : {len(files)}")

    for file_info in files:
        filename = file_info['filename']
        download_url = file_info['content_details']['download_url']
        destination = dataset_dir / filename
        download_file(download_url, destination, overwrite=overwrite)


download_mendeley_dataset(XAI4HEAT_DIR)


[mendeley] dataset: XAI4HEAT SCADA Dataset 2024
[mendeley] files  : 6
[download] https://data.mendeley.com/public-files/datasets/2mwc6x6kwb/files/31a05a1b-e1fd-4496-97be-3a89ff94b863/file_downloaded
          -> c:\Project3\HeatGrid_Agent\data\raw_data\xai4heat_scada_dataset\xai4heat_heating_area.csv
[download] https://data.mendeley.com/public-files/datasets/2mwc6x6kwb/files/8d1347aa-68ee-45b6-9239-fd0c77206082/file_downloaded
          -> c:\Project3\HeatGrid_Agent\data\raw_data\xai4heat_scada_dataset\xai4heat_scada_L12_processed.csv
[download] https://data.mendeley.com/public-files/datasets/2mwc6x6kwb/files/bb7d3b73-4a94-4fd6-be67-f904c2789215/file_downloaded
          -> c:\Project3\HeatGrid_Agent\data\raw_data\xai4heat_scada_dataset\xai4heat_scada_L17_processed.csv
[download] https://data.mendeley.com/public-files/datasets/2mwc6x6kwb/files/d9376a74-86cf-4371-9599-acd4d47124af/file_downloaded
          -> c:\Project3\HeatGrid_Agent\data\raw_data\xai4heat_scada_dataset\xai4heat_scada

## 3단계: 결과 확인

아래 셀은 실제로 파일이 저장되었는지 확인한다.

원하는 구조는 다음과 같다.

- `data/raw_data/predist_v2/...`
- `data/raw_data/xai4heat_scada_dataset/...`


In [8]:
def list_files(base_dir: Path) -> list[str]:
    return sorted(str(path.relative_to(base_dir)) for path in base_dir.rglob('*') if path.is_file())


print('[predist_v2]')
for item in list_files(PREDIST_DIR)[:100]:
    print(' -', item)

print()
print('[xai4heat_scada_dataset]')
for item in list_files(XAI4HEAT_DIR):
    print(' -', item)


[predist_v2]
 - README.md
 - manufacturer 1\configuration_types.csv
 - manufacturer 1\disturbances.csv
 - manufacturer 1\faults.csv
 - manufacturer 1\feature_descriptions.csv
 - manufacturer 1\normal_events.csv
 - manufacturer 1\operational_data\substation_1.csv
 - manufacturer 1\operational_data\substation_10.csv
 - manufacturer 1\operational_data\substation_11.csv
 - manufacturer 1\operational_data\substation_12.csv
 - manufacturer 1\operational_data\substation_13.csv
 - manufacturer 1\operational_data\substation_14.csv
 - manufacturer 1\operational_data\substation_15.csv
 - manufacturer 1\operational_data\substation_16.csv
 - manufacturer 1\operational_data\substation_17.csv
 - manufacturer 1\operational_data\substation_18.csv
 - manufacturer 1\operational_data\substation_19.csv
 - manufacturer 1\operational_data\substation_2.csv
 - manufacturer 1\operational_data\substation_20.csv
 - manufacturer 1\operational_data\substation_21.csv
 - manufacturer 1\operational_data\substation_22.